# Leaf Segmentation Inference Notebook

This notebook provides a user-friendly interface for running leaf segmentation using a trained FPN (MiT-B4) model. It is adapted from `run_inference.py`.

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
from torch.amp import autocast
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [ ]:
#this should be the name of the folder that has reconstructions or the parent folder of the folder with the extracted files 
inputSubFolderName = "/pscratch/sd/e/elavarpa/reconstructions_new"  

#outputSubfolderName = "/pscratch/sd/e/elavarpa/moth-masks" #this can be anything you want, I usually choose the current date
inputPath = os.path.join("/alsdata", inputSubFolderName)

###########################
WEIGHTS_PATH = "/pscratch/sd/e/elavarpa/model_final.pth"   # The path to where the model weights are stored
#inputPath = os.path.join(".", inputSubFolderName)

if os.path.isdir("/alsuser/pscratch"):
    wheretosave = "pscratch"
elif os.path.isdir("/alsuser/cscratch"):  
    wheretosave = "cscratch"
else:
    wheretosave = "notebooks"    

filenamelist = os.listdir(inputPath) if os.path.exists(inputPath) else []
filenamelist.sort()
for i in range(len(filenamelist)-1,np.maximum(len(filenamelist)-10000,-1),-1):
    print(f'{i}: {filenamelist[i]}')

In [ ]:
if filenamelist:
    folder_name = filenamelist[1] #update this number with the index of the file you want to process from the directory listing generated in the previous cell
    folder_path = os.path.join(inputPath, folder_name)
    print(folder_path)
    print(folder_name + 'oo')
    # outputSubfolderName = f"{folder_name}_masks"#this can be anything you want, I usually choose the current date    
    # outputPath = os.path.join("/alsuser/", wheretosave, outputSubfolderName)    
    # if not os.path.exists(outputPath):
    #     os.mkdir(outputPath)
else:
    print("No files found or path does not exist.")
    folder_path = "./sample_images"

In [ ]:
# Widget to let the user crop their data across the whole stack.
try:
    sample_img_name = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.tif', '.tiff', '.jpg', '.jpeg'))][0]
    sample_img_path = os.path.join(folder_path, sample_img_name)
    sample_img = np.array(Image.open(sample_img_path))
    height, width = sample_img.shape[:2]
    
    crop_y = widgets.IntRangeSlider(
        value=[0, height], min=0, max=height, step=1, 
        description='Y (Height):', continuous_update=False)
    
    crop_x = widgets.IntRangeSlider(
        value=[0, width], min=0, max=width, step=1, 
        description='X (Width):', continuous_update=False)
    
    def update_crop(y_range, x_range):
        cropped = sample_img[y_range[0]:y_range[1], x_range[0]:x_range[1]]
        plt.figure(figsize=(6,6))
        plt.imshow(cropped, cmap='gray')
        plt.title(f"Cropped region: shape {cropped.shape}")
        plt.axis('off')
        plt.show()
    
    out = widgets.interactive_output(update_crop, {'y_range': crop_y, 'x_range': crop_x})
    display(widgets.VBox([crop_y, crop_x, out]))
except Exception as e:
    print(f"Could not load a sample image for cropping widget: {e}")

### Data Type Discussion
Note on `inference.py` and this notebook: The image data is loaded into `float32` tensors for inference. PyTorch model inference uses `.float()` by default here. **Data from MicroCT is generally collected at float32**, but you can optionally change it to `uint8` before processing if your model was specifically trained on `uint8` scaled data (0-255). 

## Configuration
Update these paths to match your local setup.

In [ ]:
# The folder containing the images you want to segment
INPUT_IMAGES_DIR = folder_path if 'folder_path' in locals() else "./sample_images" 

# Path to the pretrained model weights provided to you
MODEL_WEIGHTS = WEIGHTS_PATH if 'WEIGHTS_PATH' in locals() else "./best_model.pth"

# Where you want the segmented masks to drop
OUTPUT_MASKS_DIR = "./output_masks"

# Where you want the fraction/porosity measurements saved
OUTPUT_CSV_FILE = "./output_fraction_results.csv"

NUM_CLASSES = 5
PATCH_SIZE = 320
STRIDE = 160

CLASS_NAMES = ["Background", "Epidermis", "Vascular_Region", "Mesophyll", "Air_Space"]

## Model Initialization

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running inference on: {device}")

# Ensure output directories exist
os.makedirs(OUTPUT_MASKS_DIR, exist_ok=True)

print("Initializing Model...")
model_eval = smp.FPN(
    encoder_name="mit_b4",
    encoder_weights=None,
    in_channels=1,
    classes=NUM_CLASSES,
)

print(f"Loading weights from: {MODEL_WEIGHTS}")
if os.path.exists(MODEL_WEIGHTS):
    checkpoint = torch.load(MODEL_WEIGHTS, map_location=device)
    model_eval.load_state_dict(checkpoint['model_state_dict'])
    model_eval = model_eval.to(device)
    model_eval.eval()
else:
    print(f"Error: Weights not found at {MODEL_WEIGHTS}")

## Inference Loop

In [ ]:
img_files = sorted([f for f in os.listdir(INPUT_IMAGES_DIR)
                    if f.lower().endswith(('.png', '.tif', '.tiff', '.jpg', '.jpeg'))])
print(f"Found {len(img_files)} images to process.")

results = []

for fname in tqdm(img_files, desc="Inference"):
    img_path = os.path.join(INPUT_IMAGES_DIR, fname)
    img = np.array(Image.open(img_path))
    
    # Apply crop if variables from widget exist
    if 'crop_y' in locals() and 'crop_x' in locals():
        y_min, y_max = crop_y.value
        x_min, x_max = crop_x.value
        img = img[y_min:y_max, x_min:x_max]
    
    # Optional: if model expects uint8 data, uncomment below to cast
    # img = (img / img.max() * 255).astype(np.uint8) if img.dtype != np.uint8 else img
    
    if img.ndim == 3 and img.shape[2] == 4:
        img = img[:, :, :3]
    if img.ndim == 3:
        img = np.dot(img[..., :3], [0.299, 0.587, 0.114]).astype(np.uint8).copy()
    
    img_t = torch.from_numpy(img.copy()).float().unsqueeze(0).unsqueeze(0)
    
    valid_pixels = img_t[img_t > 0]
    if len(valid_pixels) > 0:
        mean, std = valid_pixels.mean(), valid_pixels.std()
        if std > 1e-5:
            img_t = (img_t - mean) / std

    h, w = img.shape
    pad_h = (32 - h % 32) % 32
    pad_w = (32 - w % 32) % 32
    if pad_h > 0 or pad_w > 0:
        img_t = torch.nn.functional.pad(img_t, (0, pad_w, 0, pad_h), mode='reflect')

    img_t = img_t.to(device)
    
    _, _, H_t, W_t = img_t.shape
    pred_prob_accum = torch.zeros((1, NUM_CLASSES, H_t, W_t), device=device, dtype=torch.float32)
    count_accum     = torch.zeros((1, 1, H_t, W_t), device=device, dtype=torch.float32)
    
    with torch.no_grad():
        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        with autocast('cuda' if torch.cuda.is_available() else 'cpu', dtype=dtype):
            for py in range(0, H_t, STRIDE):
                for px in range(0, W_t, STRIDE):
                    y1 = min(py, H_t - PATCH_SIZE)
                    y2 = y1 + PATCH_SIZE
                    x1 = min(px, W_t - PATCH_SIZE)
                    x2 = x1 + PATCH_SIZE
                    
                    patch = img_t[:, :, y1:y2, x1:x2]
                    p_orig = model_eval(patch).softmax(dim=1)
                    
                    p_hflip = model_eval(torch.flip(patch, dims=[3]))
                    p_hflip = torch.flip(p_hflip.softmax(dim=1), dims=[3])
                    
                    p_vflip = model_eval(torch.flip(patch, dims=[2]))
                    p_vflip = torch.flip(p_vflip.softmax(dim=1), dims=[2])
                    
                    p_avg = (p_orig + p_hflip + p_vflip) / 3.0
                    
                    pred_prob_accum[:, :, y1:y2, x1:x2] += p_avg
                    count_accum[:, :, y1:y2, x1:x2] += 1.0
                    
    pred_prob = pred_prob_accum / count_accum
    pred_cls = torch.argmax(pred_prob, dim=1).squeeze().cpu().numpy()
            
    if pad_h > 0 or pad_w > 0:
        pred_cls = pred_cls[:h, :w]

    stem = os.path.splitext(fname)[0]
    out_path = os.path.join(OUTPUT_MASKS_DIR, f"{stem}_pred.png")
    Image.fromarray(pred_cls.astype(np.uint8)).save(out_path)
    
    total = pred_cls.size
    row = {'filename': fname}
    for c, name in enumerate(CLASS_NAMES):
        count = int(np.sum(pred_cls == c))
        row[f"{name}_pixels"] = count
        row[f"{name}_fraction"] = round(count / total, 6)
        
    row['porosity'] = row['Air_Space_fraction']
    results.append(row)

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_CSV_FILE, index=False)
print("Inference Complete!")

## Visualization

In [ ]:
if results:
    last_img_name = results[-1]['filename']
    last_mask_name = os.path.splitext(last_img_name)[0] + "_pred.png"
    
    img = Image.open(os.path.join(INPUT_IMAGES_DIR, last_img_name))
    mask = Image.open(os.path.join(OUTPUT_MASKS_DIR, last_mask_name))
    
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(img, cmap='gray')
    ax[0].set_title("Original Image")
    ax[0].axis('off')
    
    ax[1].imshow(mask, cmap='jet')
    ax[1].set_title("Predicted Mask")
    ax[1].axis('off')
    
    plt.show()
    
    display(results_df.head())